# ⚽ Predictor de Partidos — Copa del Mundo 2026

**Modelo estadístico basado en distribución de Poisson y Rankings FIFA (junio 2026)**

---
- 📊 Calcula goles esperados (λ) según diferencia de puntos FIFA
- 🎲 Distribución de Poisson para probabilidad de cada marcador posible
- 🏆 Muestra top 5 resultados más probables y probabilidades de victoria/empate
- 🎛️ App interactiva con ipywidgets

**Autor:** facuasudata | **Torneo:** FIFA World Cup 2026 (USA · México · Canadá)

In [ ]:
# ── INSTALACIÓN (solo la primera vez) ──────────────────────────
# Descomentá si no tenés alguna librería instalada
# !pip install scipy matplotlib numpy ipywidgets

In [ ]:
# ── IMPORTS ────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import poisson
import ipywidgets as widgets
from IPython.display import display, clear_output

print('✅ Librerías cargadas correctamente')

In [ ]:
# ── BASE DE DATOS: 48 CLASIFICADOS + RANKINGS FIFA JUNIO 2026 ──
# Fuente: FIFA/Coca-Cola Men's World Ranking — 11 junio 2026
# Grupos: sorteo oficial diciembre 2025

FIFA_TEAMS = {
    # UEFA (16)
    "España":              {"rank": 2,  "points": 1875, "flag": "🇪🇸", "grupo": "H", "conf": "UEFA"},
    "Francia":             {"rank": 3,  "points": 1852, "flag": "🇫🇷", "grupo": "I", "conf": "UEFA"},
    "Inglaterra":          {"rank": 4,  "points": 1820, "flag": "🏴󠁧󠁢󠁥󠁮󠁧󠁿", "grupo": "L", "conf": "UEFA"},
    "Portugal":            {"rank": 5,  "points": 1805, "flag": "🇵🇹", "grupo": "K", "conf": "UEFA"},
    "Países Bajos":        {"rank": 7,  "points": 1775, "flag": "🇳🇱", "grupo": "F", "conf": "UEFA"},
    "Bélgica":             {"rank": 9,  "points": 1745, "flag": "🇧🇪", "grupo": "G", "conf": "UEFA"},
    "Alemania":            {"rank": 10, "points": 1730, "flag": "🇩🇪", "grupo": "E", "conf": "UEFA"},
    "Suiza":               {"rank": 17, "points": 1628, "flag": "🇨🇭", "grupo": "B", "conf": "UEFA"},
    "Austria":             {"rank": 23, "points": 1570, "flag": "🇦🇹", "grupo": "J", "conf": "UEFA"},
    "Noruega":             {"rank": 24, "points": 1560, "flag": "🇳🇴", "grupo": "I", "conf": "UEFA"},
    "Turquía":             {"rank": 25, "points": 1550, "flag": "🇹🇷", "grupo": "D", "conf": "UEFA"},
    "Dinamarca":           {"rank": 22, "points": 1580, "flag": "🇩🇰", "grupo": "A", "conf": "UEFA"},
    "Suecia":              {"rank": 38, "points": 1514, "flag": "🇸🇪", "grupo": "F", "conf": "UEFA"},
    "República Checa":     {"rank": 41, "points": 1501, "flag": "🇨🇿", "grupo": "A", "conf": "UEFA"},
    "Escocia":             {"rank": 43, "points": 1498, "flag": "🏴󠁧󠁢󠁳󠁣󠁴󠁿", "grupo": "C", "conf": "UEFA"},
    "Bosnia y Herzegovina":{"rank": 65, "points": 1385, "flag": "🇧🇦", "grupo": "B", "conf": "UEFA"},
    # CONMEBOL (6)
    "Argentina":           {"rank": 1,  "points": 1895, "flag": "🇦🇷", "grupo": "J", "conf": "CONMEBOL"},
    "Brasil":              {"rank": 6,  "points": 1790, "flag": "🇧🇷", "grupo": "C", "conf": "CONMEBOL"},
    "Colombia":            {"rank": 12, "points": 1700, "flag": "🇨🇴", "grupo": "K", "conf": "CONMEBOL"},
    "Uruguay":             {"rank": 13, "points": 1685, "flag": "🇺🇾", "grupo": "H", "conf": "CONMEBOL"},
    "Ecuador":             {"rank": 21, "points": 1590, "flag": "🇪🇨", "grupo": "E", "conf": "CONMEBOL"},
    "Paraguay":            {"rank": 40, "points": 1503, "flag": "🇵🇾", "grupo": "D", "conf": "CONMEBOL"},
    # CONCACAF (6)
    "Estados Unidos":      {"rank": 17, "points": 1630, "flag": "🇺🇸", "grupo": "D", "conf": "CONCACAF"},
    "México":              {"rank": 14, "points": 1670, "flag": "🇲🇽", "grupo": "A", "conf": "CONCACAF"},
    "Canadá":              {"rank": 27, "points": 1530, "flag": "🇨🇦", "grupo": "B", "conf": "CONCACAF"},
    "Panamá":              {"rank": 33, "points": 1480, "flag": "🇵🇦", "grupo": "L", "conf": "CONCACAF"},
    "Haití":               {"rank": 81, "points": 1296, "flag": "🇭🇹", "grupo": "C", "conf": "CONCACAF"},
    "Curazao":             {"rank": 82, "points": 1293, "flag": "🇨🇼", "grupo": "E", "conf": "CONCACAF"},
    # CAF (9)
    "Marruecos":           {"rank": 8,  "points": 1760, "flag": "🇲🇦", "grupo": "C", "conf": "CAF"},
    "Senegal":             {"rank": 16, "points": 1645, "flag": "🇸🇳", "grupo": "I", "conf": "CAF"},
    "Túnez":               {"rank": 44, "points": 1483, "flag": "🇹🇳", "grupo": "F", "conf": "CAF"},
    "Argelia":             {"rank": 35, "points": 1460, "flag": "🇩🇿", "grupo": "J", "conf": "CAF"},
    "Egipto":              {"rank": 36, "points": 1450, "flag": "🇪🇬", "grupo": "G", "conf": "CAF"},
    "Costa de Marfil":     {"rank": 44, "points": 1380, "flag": "🇨🇮", "grupo": "E", "conf": "CAF"},
    "RD Congo":            {"rank": 46, "points": 1478, "flag": "🇨🇩", "grupo": "K", "conf": "CAF"},
    "Ghana":               {"rank": 73, "points": 1346, "flag": "🇬🇭", "grupo": "L", "conf": "CAF"},
    "Cabo Verde":          {"rank": 68, "points": 1369, "flag": "🇨🇻", "grupo": "H", "conf": "CAF"},
    "Sudáfrica":           {"rank": 60, "points": 1429, "flag": "🇿🇦", "grupo": "A", "conf": "CAF"},
    # AFC (6)
    "Japón":               {"rank": 15, "points": 1660, "flag": "🇯🇵", "grupo": "F", "conf": "AFC"},
    "Corea del Sur":       {"rank": 19, "points": 1615, "flag": "🇰🇷", "grupo": "A", "conf": "AFC"},
    "Irán":                {"rank": 20, "points": 1600, "flag": "🇮🇷", "grupo": "G", "conf": "AFC"},
    "Australia":           {"rank": 26, "points": 1540, "flag": "🇦🇺", "grupo": "D", "conf": "AFC"},
    "Arabia Saudita":      {"rank": 61, "points": 1419, "flag": "🇸🇦", "grupo": "H", "conf": "AFC"},
    "Qatar":               {"rank": 55, "points": 1452, "flag": "🇶🇦", "grupo": "B", "conf": "AFC"},
    "Uzbekistán":          {"rank": 50, "points": 1461, "flag": "🇺🇿", "grupo": "K", "conf": "AFC"},
    "Irak":                {"rank": 57, "points": 1447, "flag": "🇮🇶", "grupo": "I", "conf": "AFC"},
    "Jordania":            {"rank": 63, "points": 1390, "flag": "🇯🇴", "grupo": "J", "conf": "AFC"},
    # OFC (1)
    "Nueva Zelanda":       {"rank": 85, "points": 1276, "flag": "🇳🇿", "grupo": "G", "conf": "OFC"},
}

team_names = sorted(FIFA_TEAMS.keys())
print(f'✅ Base de datos cargada: {len(FIFA_TEAMS)} selecciones del Mundial 2026')
print(f'   Equipos: {len(team_names)}')

In [ ]:
# ── MODELO DE POISSON ──────────────────────────────────────────

MAX_GOALS = 7
BASE_LAMBDA = 1.35
MAX_POINTS = 1895

def calcular_lambda(pts_a, pts_b):
    diff = (pts_a - pts_b) / MAX_POINTS
    lambda_a = BASE_LAMBDA * np.exp(diff * 1.5)
    lambda_b = BASE_LAMBDA * np.exp(-diff * 1.5)
    return round(lambda_a, 4), round(lambda_b, 4)

def matriz_poisson(lambda_a, lambda_b):
    matriz = np.zeros((MAX_GOALS + 1, MAX_GOALS + 1))
    for i in range(MAX_GOALS + 1):
        for j in range(MAX_GOALS + 1):
            matriz[i][j] = poisson.pmf(i, lambda_a) * poisson.pmf(j, lambda_b)
    return matriz

def calcular_probabilidades(equipo_a, equipo_b):
    data_a = FIFA_TEAMS[equipo_a]
    data_b = FIFA_TEAMS[equipo_b]
    lA, lB = calcular_lambda(data_a['points'], data_b['points'])
    matriz = matriz_poisson(lA, lB)
    win_a  = np.sum(np.tril(matriz, -1))
    win_b  = np.sum(np.triu(matriz, 1))
    empate = np.sum(np.diag(matriz))
    scores = []
    for i in range(MAX_GOALS + 1):
        for j in range(MAX_GOALS + 1):
            scores.append((f"{i}-{j}", round(matriz[i][j] * 100, 2)))
    scores.sort(key=lambda x: x[1], reverse=True)
    return {
        'win_a': round(win_a * 100, 1),
        'win_b': round(win_b * 100, 1),
        'empate': round(empate * 100, 1),
        'lambda_a': lA, 'lambda_b': lB,
        'top5': scores[:5],
        'mejor_resultado': scores[0],
        'matriz': matriz,
    }

print('✅ Modelo de Poisson listo')

In [ ]:
# ── VISUALIZACIÓN ──────────────────────────────────────────────

def graficar_resultado(equipo_a, equipo_b, res):
    da = FIFA_TEAMS[equipo_a]
    db = FIFA_TEAMS[equipo_b]
    fig = plt.figure(figsize=(14, 9))
    fig.patch.set_facecolor('#0a1628')
    gs = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35,
                          left=0.06, right=0.97, top=0.88, bottom=0.08)
    fig.text(0.5, 0.95, f'⚽  {da["flag"]} {equipo_a}  vs  {db["flag"]} {equipo_b}',
             ha='center', va='top', fontsize=18, fontweight='bold', color='white')
    fig.text(0.5, 0.91,
             f'Grupo {da["grupo"]}  ·  Rankings FIFA: #{da["rank"]} vs #{db["rank"]}  ·  Copa del Mundo 2026',
             ha='center', va='top', fontsize=10, color='#7ba7c4')
    DARK=  '#0a1628'; MID= '#0d2137'; GOLD= '#c9a227'
    BLUE=  '#3b82f6'; RED= '#ef4444'; LIGHT='#e8f4fd'; MUTED='#7ba7c4'

    # 1. Barras probabilidad
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.set_facecolor(MID)
    for s in ax1.spines.values(): s.set_visible(False)
    labels  = [f'{da["flag"]} {equipo_a}', 'Empate', f'{db["flag"]} {equipo_b}']
    valores = [res['win_a'], res['empate'], res['win_b']]
    colores = [BLUE, GOLD, RED]
    bars = ax1.barh(labels, valores, color=colores, height=0.5, edgecolor='none')
    for bar, val in zip(bars, valores):
        ax1.text(bar.get_width()+0.8, bar.get_y()+bar.get_height()/2,
                 f'{val:.1f}%', va='center', ha='left', fontsize=11, fontweight='bold', color=LIGHT)
    ax1.set_xlim(0, max(valores)+14)
    ax1.set_xlabel('Probabilidad (%)', color=MUTED, fontsize=9)
    ax1.set_title('Probabilidades', color=LIGHT, fontsize=11, fontweight='bold', pad=10)
    ax1.tick_params(colors=LIGHT, labelsize=9)

    # 2. Top 5 marcadores
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.set_facecolor(MID)
    for s in ax2.spines.values(): s.set_visible(False)
    marcadores   = [s[0] for s in res['top5']]
    probs_marc   = [s[1] for s in res['top5']]
    colores_marc = [GOLD] + ['#2563eb'] * 4
    bars2 = ax2.bar(marcadores, probs_marc, color=colores_marc, edgecolor='none', width=0.6)
    for bar, val in zip(bars2, probs_marc):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold', color=LIGHT)
    ax2.set_ylabel('Probabilidad (%)', color=MUTED, fontsize=9)
    ax2.set_title('Top 5 marcadores más probables', color=LIGHT, fontsize=11, fontweight='bold', pad=10)
    ax2.tick_params(colors=LIGHT, labelsize=9)
    ax2.set_ylim(0, max(probs_marc)*1.3)
    gold_patch = mpatches.Patch(color=GOLD, label='Más probable')
    ax2.legend(handles=[gold_patch], fontsize=8, facecolor=MID, labelcolor=LIGHT, edgecolor='none')

    # 3. Distribución Poisson
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.set_facecolor(MID)
    for s in ax3.spines.values(): s.set_visible(False)
    goles = np.arange(0, 7)
    dist_a = [poisson.pmf(g, res['lambda_a'])*100 for g in goles]
    dist_b = [poisson.pmf(g, res['lambda_b'])*100 for g in goles]
    x = np.arange(len(goles)); width = 0.35
    ax3.bar(x-width/2, dist_a, width, label=equipo_a, color=BLUE, alpha=0.85, edgecolor='none')
    ax3.bar(x+width/2, dist_b, width, label=equipo_b, color=RED,  alpha=0.85, edgecolor='none')
    ax3.set_xticks(x); ax3.set_xticklabels([str(g) for g in goles])
    ax3.set_xlabel('Goles', color=MUTED, fontsize=9)
    ax3.set_ylabel('Probabilidad (%)', color=MUTED, fontsize=9)
    ax3.set_title('Distribución de goles (Poisson)', color=LIGHT, fontsize=11, fontweight='bold', pad=10)
    ax3.tick_params(colors=LIGHT, labelsize=9)
    ax3.legend(fontsize=8, facecolor=MID, labelcolor=LIGHT, edgecolor='none')

    # 4. Heatmap
    ax4 = fig.add_subplot(gs[1, :])
    ax4.set_facecolor(MID)
    for s in ax4.spines.values(): s.set_visible(False)
    n = 6
    heatmap_data = res['matriz'][:n+1, :n+1] * 100
    im = ax4.imshow(heatmap_data, cmap='YlOrRd', aspect='auto', vmin=0, vmax=heatmap_data.max())
    for i in range(n+1):
        for j in range(n+1):
            val = heatmap_data[i, j]
            ct  = 'black' if val > heatmap_data.max()*0.5 else LIGHT
            ax4.text(j, i, f'{val:.1f}%', ha='center', va='center', fontsize=8.5, color=ct, fontweight='bold')
    ax4.set_xticks(range(n+1)); ax4.set_yticks(range(n+1))
    ax4.set_xticklabels([str(i) for i in range(n+1)], color=LIGHT, fontsize=9)
    ax4.set_yticklabels([str(i) for i in range(n+1)], color=LIGHT, fontsize=9)
    ax4.set_xlabel(f'Goles {db["flag"]} {equipo_b}', color=MUTED, fontsize=10, labelpad=8)
    ax4.set_ylabel(f'Goles {da["flag"]} {equipo_a}', color=MUTED, fontsize=10, labelpad=8)
    ax4.set_title('Mapa de probabilidad por marcador exacto', color=LIGHT, fontsize=11, fontweight='bold', pad=10)
    cbar = plt.colorbar(im, ax=ax4, orientation='vertical', pad=0.01, fraction=0.015)
    cbar.ax.tick_params(colors=LIGHT, labelsize=8)
    cbar.set_label('%', color=LIGHT, fontsize=9)

    fname = f'{equipo_a.replace(" ","_")}_vs_{equipo_b.replace(" ","_")}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor=DARK)
    plt.show()
    print(f'\n💾 Gráfico guardado: {fname}')

print('✅ Funciones de visualización listas')

In [ ]:
# ── APP INTERACTIVA ────────────────────────────────────────────

style  = {'description_width': '80px'}
layout = widgets.Layout(width='280px')

dd_a = widgets.Dropdown(
    options=team_names, value='Brasil',
    description='🏠 Equipo A:', style=style, layout=layout
)
dd_b = widgets.Dropdown(
    options=team_names, value='Marruecos',
    description='✈️ Equipo B:', style=style, layout=layout
)
btn = widgets.Button(
    description='⚽ Calcular probabilidades',
    button_style='warning',
    layout=widgets.Layout(width='280px', height='40px')
)
out = widgets.Output()

def on_click(b):
    with out:
        clear_output(wait=True)
        ea = dd_a.value; eb = dd_b.value
        if ea == eb:
            print('⚠️  Seleccioná dos equipos distintos.')
            return
        da = FIFA_TEAMS[ea]; db = FIFA_TEAMS[eb]
        res = calcular_probabilidades(ea, eb)
        sep = '─' * 50
        print(f'\n{sep}')
        print(f'  {da["flag"]} {ea:25s} Ranking #{da["rank"]} · {da["conf"]}')
        print(f'  {db["flag"]} {eb:25s} Ranking #{db["rank"]} · {db["conf"]}')
        print(f'{sep}')
        print(f'  λ esperados → {ea}: {res["lambda_a"]}   {eb}: {res["lambda_b"]}')
        print(f'{sep}')
        print(f'  🔵 {ea} gana:  {res["win_a"]:5.1f}%')
        print(f'  🟡 Empate:      {res["empate"]:5.1f}%')
        print(f'  🔴 {eb} gana:  {res["win_b"]:5.1f}%')
        print(f'{sep}')
        print(f'  🏆 Resultado más probable: {res["mejor_resultado"][0]} ({res["mejor_resultado"][1]:.1f}%)')
        print(f'\n  Top 5 marcadores:')
        for i, (score, prob) in enumerate(res['top5'], 1):
            bar = '█' * int(prob / 0.8)
            print(f'  {i}. {score:6s}  {prob:4.1f}%  {bar}')
        print(f'{sep}\n')
        graficar_resultado(ea, eb, res)

btn.on_click(on_click)

display(
    widgets.HTML('<h3 style="color:#c9a227">⚽ Predictor — Copa del Mundo 2026</h3>'),
    widgets.HTML('<p style="color:#7ba7c4;font-size:12px">48 equipos · Modelo de Poisson · Rankings FIFA 11 junio 2026</p>'),
    dd_a, dd_b, btn, out
)
btn.click()

---
## 📝 Cómo funciona el modelo

### Distribución de Poisson
$$P(X = k) = \frac{e^{-\lambda} \cdot \lambda^k}{k!}$$

### Cálculo de λ (goles esperados)
$$\lambda_A = 1.35 \cdot e^{\frac{pts_A - pts_B}{pts_{max}} \cdot 1.5}$$

### Fuente
Rankings FIFA/Coca-Cola Men's World Ranking — **11 de junio 2026**  
48 equipos clasificados — grupos oficiales del sorteo de diciembre 2025

---
*Proyecto de portfolio · facuasudata · GitHub*